# Encoder-Only Transformer: Complete Pipeline Lab

## Full Pipeline

### Training
`Text Corpus → Tokenization → Embedding → Positional Encoding → Encoder Stack → Training Head → Loss → Backpropagation → Saved Weights (.pth/.bin/.safetensors)`

### Inference
`User Text → Tokenization → Embedding → Positional Encoding → Encoder Stack → Output Representation / Embedding / Classification`

---

**Examples:** BERT, RoBERTa, ALBERT, DistilBERT

**Key difference from decoder/encoder-decoder:** Encoder-only models **understand** text (classification, NER, similarity) — they don't **generate** text.

We will build each stage one by one.

---
## Step 1: Text Corpus

### What?
Raw text data the model learns from. Unlike encoder-decoder (which needs source-target pairs), encoder-only models learn from **unlabeled text** during pre-training, and **labeled text** during fine-tuning.

### Why?
- Pre-training: The model learns language structure via self-supervised tasks like **Masked Language Modeling (MLM)** — predict masked words from context.
- Fine-tuning: The model learns task-specific patterns from labeled data (e.g., sentiment labels, NER tags).

### Where in the pipeline?
```
👉 [Text Corpus] → Tokenization → Embedding → Positional Encoding → Encoder Stack → Training Head → Loss
```

### Encoder-only vs Encoder-Decoder data
| | Encoder-Only | Encoder-Decoder |
|--|-------------|----------------|
| Pre-training data | Unlabeled text | Parallel pairs |
| Pre-training task | MLM (fill in blanks) | Seq-to-seq reconstruction |
| Fine-tuning data | (text, label) | (source, target) |
| Fine-tuning task | Classification, NER, etc. | Translation, summarization |

In [ ]:
# Step 1: Text Corpus — Labeled Data for Sentiment Classification

# For this lab, we'll fine-tune an encoder-only model for sentiment classification.
# Each sample: (text, label)  where label: 0=negative, 1=positive

corpus = [
    # (text, label)
    ("I love this movie it is amazing", 1),
    ("This film is wonderful and great", 1),
    ("Absolutely fantastic experience", 1),
    ("Best movie I have ever seen", 1),
    ("This movie is terrible and boring", 0),
    ("Worst film I have ever watched", 0),
    ("Absolutely awful waste of time", 0),
    ("I hate this movie it is bad", 0),
]

label_names = {0: 'negative', 1: 'positive'}

print(f"Corpus size: {len(corpus)} samples")
print(f"Labels: {label_names}")
print(f"\nExample:")
print(f"  Text:  '{corpus[0][0]}'")
print(f"  Label: {corpus[0][1]} ({label_names[corpus[0][1]]})")

### Key Takeaway

- Encoder-only models work with **(text, label)** pairs for fine-tuning
- No target sequence to generate — just classify/tag/embed the input
- The model reads the **entire** input bidirectionally (unlike decoder which is causal)

**Next Step →** Tokenization: converting text into token IDs.

---
## Step 2: Tokenization

### What?
Converting raw text into **token IDs**. For encoder-only models like BERT, tokenization also adds special tokens:
- `[CLS]` at the start — its output embedding becomes the **sentence-level representation** used for classification
- `[SEP]` at the end — marks sentence boundary

### Why?
- Neural networks need numbers, not strings.
- `[CLS]` token is critical — it **aggregates information from all tokens** via self-attention and becomes the single vector used for classification.
- Unlike decoder models, there's **no `<sos>`** — encoder-only doesn't generate tokens.

### Where in the pipeline?
```
Text Corpus → 👉 [Tokenization] → Embedding → Positional Encoding → Encoder Stack → Training Head
```

### Special Tokens
| Token | ID | Purpose |
|-------|----|---------|
| `[PAD]` | 0 | Padding shorter sequences |
| `[CLS]` | 1 | Classification token — sentence representation |
| `[SEP]` | 2 | Separator / end of sentence |
| `[UNK]` | 3 | Unknown token |
| `[MASK]` | 4 | Masked token (used in MLM pre-training) |

In [ ]:
# Step 2: Tokenization — Building Vocabulary & Converting Text to IDs

import torch
import torch.nn as nn
import math

# Special tokens (BERT-style)
PAD_TOKEN, CLS_TOKEN, SEP_TOKEN, UNK_TOKEN, MASK_TOKEN = '[PAD]', '[CLS]', '[SEP]', '[UNK]', '[MASK]'
PAD_IDX, CLS_IDX, SEP_IDX, UNK_IDX, MASK_IDX = 0, 1, 2, 3, 4

class Tokenizer:
    """Word-level tokenizer with BERT-style special tokens."""
    
    def __init__(self):
        self.word2idx = {PAD_TOKEN: 0, CLS_TOKEN: 1, SEP_TOKEN: 2, UNK_TOKEN: 3, MASK_TOKEN: 4}
        self.idx2word = {v: k for k, v in self.word2idx.items()}
    
    def build_vocab(self, sentences):
        for sent in sentences:
            for word in sent.lower().split():
                if word not in self.word2idx:
                    idx = len(self.word2idx)
                    self.word2idx[word] = idx
                    self.idx2word[idx] = word
        print(f"Vocab size: {len(self.word2idx)}")
    
    def encode(self, sentence):
        """Encode with [CLS] ... [SEP] wrapping."""
        ids = [self.word2idx.get(w, UNK_IDX) for w in sentence.lower().split()]
        return [CLS_IDX] + ids + [SEP_IDX]
    
    def decode(self, ids):
        return ' '.join(self.idx2word.get(i, UNK_TOKEN) for i in ids)


tokenizer = Tokenizer()
tokenizer.build_vocab([text for text, label in corpus])

# Tokenize one example
text, label = corpus[0]
token_ids = tokenizer.encode(text)

print(f"\nText:      '{text}'")
print(f"Token IDs: {token_ids}")
print(f"Decoded:   {tokenizer.decode(token_ids)}")
print(f"\n[CLS] at position 0 → its output will be used for classification")
print(f"[SEP] at the end   → marks sentence boundary")

### Key Takeaway

- Encoder-only uses `[CLS] tokens... [SEP]` format (no `<sos>` — not generating)
- `[CLS]` token's output embedding = **sentence representation** for classification
- Single vocabulary (not separate src/tgt like encoder-decoder)

**Next Step →** Embedding: converting token IDs into dense vectors.

---
## Step 3: Embedding

### What?
A learnable lookup table mapping each token ID to a dense vector of size `d_model`. For BERT-style models, the embedding is actually a **sum of three embeddings**:
1. **Token Embedding** — semantic meaning of the token
2. **Position Embedding** — where the token is in the sequence (often learned, not sinusoidal)
3. **Segment Embedding** — which sentence the token belongs to (for sentence-pair tasks)

For simplicity, we'll use token embedding + sinusoidal positional encoding (next step).

### Why?
- Token IDs are arbitrary integers — no semantic meaning.
- Embeddings place tokens in continuous space where similar words are nearby.
- Only **one** embedding layer needed (single vocabulary, unlike encoder-decoder).

### Where in the pipeline?
```
Text Corpus → Tokenization → 👉 [Embedding] → Positional Encoding → Encoder Stack → Training Head
```

In [ ]:
# Step 3: Embedding — Token IDs → Dense Vectors

class TokenEmbedding(nn.Module):
    """Converts token IDs to scaled dense vectors."""
    
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.d_model = d_model
    
    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)


d_model = 256  # smaller than original BERT (768) for demo
vocab_size = len(tokenizer.word2idx)

embedding = TokenEmbedding(vocab_size, d_model)

# Convert tokenized example to tensor
input_tensor = torch.tensor([token_ids])  # (1, seq_len)
embedded = embedding(input_tensor)          # (1, seq_len, d_model)

print(f"Vocab size: {vocab_size}")
print(f"Input tensor: {input_tensor} → shape {input_tensor.shape}")
print(f"Embedded shape: {embedded.shape} → (batch, seq_len, d_model)")
print(f"\n[CLS] embedding (first 10 dims): {embedded[0, 0, :10].detach()}")

### Key Takeaway

- Single embedding layer (one vocabulary, one language)
- Scaled by `√(d_model)` to balance with positional encoding
- Output: `(batch, seq_len, d_model)` — each token is now a dense vector

**Next Step →** Positional Encoding: adding position information.

---
## Step 4: Positional Encoding

### What?
Adds position information to embeddings using sinusoidal functions. Without this, the model treats "dog bites man" and "man bites dog" identically.

### Why?
- Transformers process all tokens **in parallel** — no inherent notion of order.
- Position matters for understanding: "not good" ≠ "good not" in meaning.

### Encoder-only specifics
- **No causal mask** — encoder-only models are **bidirectional** (every token sees every other token).
- This is the key advantage: `[CLS]` can attend to the **entire** sentence in both directions.
- BERT uses **learned** positional embeddings; we use sinusoidal for simplicity (same concept).

### Where in the pipeline?
```
Text Corpus → Tokenization → Embedding → 👉 [Positional Encoding] → Encoder Stack → Training Head
```

In [ ]:
# Step 4: Positional Encoding — Injecting Order Information

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


pos_encoder = PositionalEncoding(d_model, dropout=0.1)
pos_encoded = pos_encoder(embedded)

print(f"After positional encoding: {pos_encoded.shape}")
print(f"\nBefore PE (first 5 dims): {embedded[0, 0, :5].detach()}")
print(f"After  PE (first 5 dims): {pos_encoded[0, 0, :5].detach()}")
print(f"\nKey: NO causal mask — encoder-only is BIDIRECTIONAL.")

### Key Takeaway

- Positional encoding is **added** to embeddings (same as encoder-decoder)
- Encoder-only is **bidirectional** — no causal mask, every token sees everything
- This is why BERT is great for **understanding** tasks (classification, NER, similarity)

**Next Step →** Encoder Stack: building deep contextualized representations.

---
## Step 5: Encoder Stack

### What?
A stack of N identical layers, each containing:
1. **Multi-Head Self-Attention** — every token attends to every other token (bidirectional!)
2. **Feed-Forward Network** — position-wise non-linear transformation
3. **Residual connections + Layer Normalization** around each sub-layer

### Why?
- Self-attention builds **contextualized representations** — same word gets different vectors in different contexts.
- **Bidirectional** attention (no mask!) means the model sees the full context in both directions.
- Stacking layers = deeper understanding. Layer 1 captures syntax, deeper layers capture semantics.

### Where in the pipeline?
```
Text Corpus → Tokenization → Embedding → PE → 👉 [Encoder Stack] → Training Head → Loss
```

### Encoder-only vs Decoder: The Masking Difference
| | Encoder-Only | Decoder-Only |
|--|-------------|-------------|
| Attention | **Bidirectional** (no mask) | **Causal** (masked) |
| Token 3 sees | Tokens 1,2,3,4,5... | Tokens 1,2,3 only |
| Good for | Understanding | Generation |

In [ ]:
# Step 5: Encoder Stack

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        Q = self.W_q(query).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = self.dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, -1, self.n_heads * self.d_k)
        return self.W_o(out)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=1024, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, src, mask=None):
        # NO CAUSAL MASK — bidirectional self-attention
        attn_out = self.self_attn(src, src, src, mask)
        src = self.norm1(src + self.dropout1(attn_out))
        ffn_out = self.ffn(src)
        src = self.norm2(src + self.dropout2(ffn_out))
        return src


class Encoder(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, src, mask=None):
        for layer in self.layers:
            src = layer(src, mask)
        return self.norm(src)


# Config
n_layers = 4
n_heads = 4
d_ff = 1024
dropout = 0.1

encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)

encoder.eval()
with torch.no_grad():
    encoder_output = encoder(pos_encoded)  # NO mask passed → bidirectional!

print(f"Encoder input shape:  {pos_encoded.shape}")
print(f"Encoder output shape: {encoder_output.shape}")
print(f"\n[CLS] token output (position 0): {encoder_output[0, 0, :5].detach()}")
print(f"This [CLS] vector now contains information from ALL tokens (bidirectional).")

### Key Takeaway

- Encoder stack = N layers of (Self-Attention + FFN) — same as encoder-decoder's encoder
- **No causal mask** → fully bidirectional attention
- `[CLS]` output at position 0 aggregates the entire sentence's meaning
- Output shape unchanged: `(batch, seq_len, d_model)` but deeply contextualized

**Next Step →** Training Head: using the encoder output for classification.

---
## Step 6: Training Head

### What?
A task-specific layer on top of the encoder output. The head depends on the task:

| Task | Head | Input | Output |
|------|------|-------|--------|
| Classification | Linear(d_model, n_classes) | `[CLS]` output | Class probabilities |
| NER / Token classification | Linear(d_model, n_tags) | All token outputs | Tag per token |
| MLM (pre-training) | Linear(d_model, vocab_size) | Masked token outputs | Predicted tokens |
| Sentence similarity | Cosine similarity | `[CLS]` from two sentences | Similarity score |

### Why?
- The encoder produces **general-purpose representations** — the head adapts them to a specific task.
- For classification: we only need the `[CLS]` token's output (it has seen the whole sentence).
- The head is typically a simple linear layer — the encoder does the heavy lifting.

### Where in the pipeline?
```
Text Corpus → Tokenization → Embedding → PE → Encoder Stack → 👉 [Training Head] → Loss
```

In [ ]:
# Step 6: Training Head — Classification Head

class ClassificationHead(nn.Module):
    """Takes [CLS] output and predicts class."""
    
    def __init__(self, d_model, n_classes, dropout=0.1):
        super().__init__()
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes)
        )
    
    def forward(self, encoder_output):
        # Take [CLS] token output (position 0)
        cls_output = encoder_output[:, 0, :]  # (batch, d_model)
        return self.head(cls_output)            # (batch, n_classes)


n_classes = 2  # positive / negative
cls_head = ClassificationHead(d_model, n_classes)

with torch.no_grad():
    logits = cls_head(encoder_output)

probs = torch.softmax(logits, dim=-1)
predicted = torch.argmax(probs, dim=-1)

print(f"Encoder output shape: {encoder_output.shape}")
print(f"[CLS] vector shape:   {encoder_output[:, 0, :].shape}")
print(f"Logits:               {logits[0].detach()}")
print(f"Probabilities:        {probs[0].detach()}")
print(f"Predicted class:      {predicted.item()} ({label_names[predicted.item()]})")
print(f"\n(Random prediction since model is untrained)")

### Key Takeaway

- Classification head uses **only `[CLS]` output** (position 0) — not all tokens
- `[CLS]` has attended to every token bidirectionally → it represents the whole sentence
- Head is simple: `Linear → Tanh → Linear` — the encoder does the real work
- Output: `(batch, n_classes)` → one prediction per input

**Next Step →** Loss: measuring classification error.

---
## Step 7: Loss

### What?
**Cross-Entropy Loss** for classification — measures how far the predicted class probabilities are from the true label.

### Why?
- We need a single number to tell the model how wrong it is.
- CrossEntropy is the standard for classification: `-log(probability of correct class)`.
- If model is confident and correct → low loss. If wrong → high loss.

### Encoder-only vs Encoder-Decoder loss
| | Encoder-Only (Classification) | Encoder-Decoder (Seq2Seq) |
|--|-------------------------------|--------------------------|
| Loss computed on | `[CLS]` output vs label | Every target token vs prediction |
| Shape | `(batch, n_classes)` vs `(batch,)` | `(batch*tgt_len, vocab)` vs `(batch*tgt_len,)` |
| One loss per | Sentence | Token |

### Where in the pipeline?
```
... → Encoder Stack → Training Head → 👉 [Loss] → Backpropagation
```

In [ ]:
# Step 7: Loss — Measuring Classification Error

criterion = nn.CrossEntropyLoss()

# Single example
label_tensor = torch.tensor([label])  # (1,) → actual label

loss = criterion(logits, label_tensor)

print(f"Logits:          {logits[0].detach()} → (n_classes={n_classes})")
print(f"True label:      {label} ({label_names[label]})")
print(f"Loss:            {loss.item():.4f}")
print(f"\n(High loss expected for untrained model — random predictions)")

### Key Takeaway

- CrossEntropyLoss for classification: `(batch, n_classes)` logits vs `(batch,)` labels
- Much simpler than seq2seq loss — one prediction per sentence, not per token
- No `ignore_index` needed (no padding in labels)

**Next Step →** Backpropagation: training the full model.

---
## Step 8: Backpropagation & Training

### What?
Compute gradients of the loss w.r.t. all parameters, then update weights using the optimizer.

### Where in the pipeline?
```
... → Training Head → Loss → 👉 [Backpropagation] → Saved Weights
```

### Training loop
```
for each epoch:
    for each batch:
        1. Tokenize + pad → input tensor
        2. Embed → PE → Encoder → [CLS] → Classification Head → logits
        3. Loss = CrossEntropy(logits, labels)
        4. loss.backward()
        5. optimizer.step()
```

In [ ]:
# Step 8: Backpropagation — Full Training Loop

# Assemble the full Encoder-Only model
class EncoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, n_classes, dropout=0.1):
        super().__init__()
        self.embedding = TokenEmbedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, dropout=dropout)
        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)
        self.cls_head = ClassificationHead(d_model, n_classes, dropout)
    
    def forward(self, src, mask=None):
        x = self.pos_encoding(self.embedding(src))
        encoder_out = self.encoder(x, mask)
        return self.cls_head(encoder_out)  # (batch, n_classes)
    
    def get_embeddings(self, src, mask=None):
        """Get [CLS] embeddings for inference (similarity, retrieval)."""
        x = self.pos_encoding(self.embedding(src))
        encoder_out = self.encoder(x, mask)
        return encoder_out[:, 0, :]  # [CLS] output


# Prepare batched data
def prepare_batch(corpus, tokenizer):
    all_ids, all_labels = [], []
    for text, label in corpus:
        all_ids.append(tokenizer.encode(text))
        all_labels.append(label)
    max_len = max(len(ids) for ids in all_ids)
    padded = torch.tensor([ids + [PAD_IDX] * (max_len - len(ids)) for ids in all_ids])
    labels = torch.tensor(all_labels)
    # Padding mask: True where NOT padded
    pad_mask = (padded != PAD_IDX).unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, seq_len)
    return padded, labels, pad_mask


# Build model
model = EncoderOnlyTransformer(
    vocab_size=vocab_size, d_model=d_model, n_heads=n_heads,
    n_layers=n_layers, d_ff=d_ff, n_classes=n_classes, dropout=dropout
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

input_batch, label_batch, pad_mask = prepare_batch(corpus, tokenizer)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Input batch: {input_batch.shape}")
print(f"Labels: {label_batch.tolist()}")

# Training loop
model.train()
n_epochs = 200

for epoch in range(n_epochs):
    logits = model(input_batch, pad_mask)
    loss = criterion(logits, label_batch)
    
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    
    if (epoch + 1) % 40 == 0:
        acc = (logits.argmax(dim=-1) == label_batch).float().mean()
        print(f"Epoch {epoch+1:3d}/{n_epochs} | Loss: {loss.item():.4f} | Accuracy: {acc.item():.2%}")

print(f"\nTraining complete! Final loss: {loss.item():.4f}")

### Key Takeaway

- Full model: Embedding + PE + Encoder + Classification Head
- **Padding mask** ensures padded tokens don't affect attention scores
- Training loop: forward → loss → backward → clip → step
- Accuracy should reach ~100% on this small dataset

**Next Step →** Saved Weights: persisting the trained model.

---
## Step 9: Saved Weights (.pth / .bin / .safetensors)

### What?
Save the trained model's weights, config, and tokenizer to disk.

### Where in the pipeline?
```
... → Loss → Backpropagation → 👉 [Saved Weights] (.pth/.bin/.safetensors)
```

In [ ]:
# Step 9: Saving the Trained Model
import json, os

save_dir = 'trained_encoder_only_model'
os.makedirs(save_dir, exist_ok=True)

# 1. Save weights
torch.save(model.state_dict(), f'{save_dir}/model.pth')

# 2. Save config
config = {
    'vocab_size': vocab_size, 'd_model': d_model, 'n_heads': n_heads,
    'n_layers': n_layers, 'd_ff': d_ff, 'n_classes': n_classes, 'dropout': dropout
}
with open(f'{save_dir}/config.json', 'w') as f:
    json.dump(config, f, indent=2)

# 3. Save tokenizer vocab
with open(f'{save_dir}/vocab.json', 'w') as f:
    json.dump(tokenizer.word2idx, f, indent=2)

# 4. Save label mapping
with open(f'{save_dir}/labels.json', 'w') as f:
    json.dump(label_names, f, indent=2)

print(f"Model saved to '{save_dir}/':")
for f in os.listdir(save_dir):
    size = os.path.getsize(f'{save_dir}/{f}')
    print(f"  {f:20s} ({size:,} bytes)")

### Key Takeaway

- Saved model = **Weights + Config + Vocab + Labels**
- This completes the **Training Pipeline**!

---

# 🚀 Inference Pipeline

`User Text → Tokenization → Embedding → Positional Encoding → Encoder Stack → Output Representation / Embedding / Classification`

**Next Step →** Inference: loading the model and classifying new text.

---
## Step 10: Inference — Classification & Embeddings

### What?
Load the saved model and use it for:
1. **Classification** — predict sentiment of new text
2. **Embeddings** — get sentence representations for similarity/retrieval

### How inference differs from training
| | Training | Inference |
|--|---------|-----------|
| Mode | `model.train()` | `model.eval()` + `torch.no_grad()` |
| Dropout | Active (regularization) | Disabled |
| Gradients | Computed | Not needed |
| Output | Loss for backprop | Predictions / embeddings |

### Where in the pipeline?
```
User Text → Tokenize → Embed → PE → Encoder → [CLS] output
                                                    │
                                        ┌────────────┼────────────┐
                                        │            │            │
                                  Classification  Embedding   Similarity
```

### Key: No autoregressive loop!
Unlike decoder models, encoder-only runs **once** per input — no token-by-token generation.

In [ ]:
# Step 10: Inference — Load Model & Classify / Get Embeddings

# --- Load saved model ---
with open(f'{save_dir}/config.json') as f:
    cfg = json.load(f)
with open(f'{save_dir}/labels.json') as f:
    lbl_names = {int(k): v for k, v in json.load(f).items()}

# Rebuild tokenizer
inf_tokenizer = Tokenizer()
with open(f'{save_dir}/vocab.json') as f:
    inf_tokenizer.word2idx = json.load(f)
    inf_tokenizer.idx2word = {int(v): k for k, v in inf_tokenizer.word2idx.items()}

# Rebuild and load model
inf_model = EncoderOnlyTransformer(**cfg)
inf_model.load_state_dict(torch.load(f'{save_dir}/model.pth', weights_only=True))
inf_model.eval()
print(f"Model loaded from '{save_dir}/'")

In [ ]:
# --- Task 1: Classification ---

@torch.no_grad()
def classify(model, text, tokenizer, label_names):
    ids = torch.tensor([tokenizer.encode(text)])
    logits = model(ids)
    probs = torch.softmax(logits, dim=-1)
    pred = logits.argmax(dim=-1).item()
    return label_names[pred], probs[0][pred].item()


print('=== Sentiment Classification ===\n')
test_texts = [
    'I love this movie it is amazing',    # from training (positive)
    'This movie is terrible and boring',   # from training (negative)
    'I love this wonderful film',          # unseen (positive)
    'Terrible awful movie',                # unseen (negative)
]

for text in test_texts:
    sentiment, confidence = classify(inf_model, text, inf_tokenizer, lbl_names)
    print(f"  '{text}'")
    print(f"  → {sentiment} (confidence: {confidence:.2%})\n")

In [ ]:
# --- Task 2: Sentence Embeddings (for similarity / retrieval) ---

@torch.no_grad()
def get_embedding(model, text, tokenizer):
    ids = torch.tensor([tokenizer.encode(text)])
    return model.get_embeddings(ids)  # [CLS] output


def cosine_similarity(a, b):
    return torch.nn.functional.cosine_similarity(a, b).item()


print('=== Sentence Similarity (via [CLS] embeddings) ===\n')

pairs = [
    ('I love this movie', 'This film is wonderful'),
    ('I love this movie', 'I hate this movie'),
    ('I love this movie', 'Absolutely awful waste of time'),
]

for text_a, text_b in pairs:
    emb_a = get_embedding(inf_model, text_a, inf_tokenizer)
    emb_b = get_embedding(inf_model, text_b, inf_tokenizer)
    sim = cosine_similarity(emb_a, emb_b)
    print(f"  '{text_a}'  vs  '{text_b}'")
    print(f"  → Cosine similarity: {sim:.4f}\n")

### Key Takeaway

- Inference is a **single forward pass** — no autoregressive loop
- `model.eval()` + `torch.no_grad()` for inference (disables dropout, saves memory)
- **Classification**: `[CLS]` → head → softmax → predicted class
- **Embeddings**: `[CLS]` output directly → use for similarity, retrieval, clustering

---

## 🎯 Summary: Complete Encoder-Only Pipeline

### Training Pipeline
```
Text Corpus                 → (text, label) pairs
  → Tokenization            → [CLS] tokens... [SEP] + padding
  → Embedding               → Token IDs → dense vectors (d_model)
  → Positional Encoding     → Add position info (sinusoidal)
  → Encoder Stack           → Bidirectional self-attention + FFN (N layers)
  → Training Head           → [CLS] output → Linear → class logits
  → Loss                    → CrossEntropy(predicted vs actual label)
  → Backpropagation         → Gradients → optimizer → update weights
  → Saved Weights           → .pth + config.json + vocab + labels
```

### Inference Pipeline
```
User Text                   → Raw input
  → Tokenization            → [CLS] tokens... [SEP]
  → Embedding + PE          → Dense vectors with position
  → Encoder Stack           → Bidirectional contextualization
  → Output                  → Classification / Embedding / Similarity
```

### Key Differences from Decoder / Encoder-Decoder
| Feature | Encoder-Only | Decoder-Only | Encoder-Decoder |
|---------|-------------|-------------|----------------|
| Attention | Bidirectional | Causal (masked) | Both |
| Task | Understanding | Generation | Seq-to-Seq |
| Inference | Single pass | Autoregressive loop | Encode once + decode loop |
| Output | Embeddings / classes | Generated text | Generated text |
| Examples | BERT, RoBERTa | GPT | T5, BART |